# Load Data

In [1]:
import pandas as pd
import numpy as np
import spotipy
from spotipy.oauth2 import SpotifyOAuth
from dotenv import load_dotenv
import os
from pathlib import Path

# Extract credential access Spotify API

In [2]:
env_path = Path("..") / ".env"
load_dotenv(dotenv_path=env_path)
CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")
REDIRECT_URI = os.getenv("REDIRECT_URI")

# Extract Spotify data

## Helper Functions

In [3]:
def extract_top_tracks_data(spotify_client,time_range)->pd.DataFrame:
    '''
    Convert JSON file extracted from spotify 
    to dataframe for analysis
    '''
    top_tracks_json = spotify_client.current_user_top_tracks(limit=50, time_range=time_range)
    track_list = []

    for item in top_tracks_json['items']:
        track_info = {
            'track_name':item['name'],
            'artist_name': item['artists'][0]['name'],
            'album_name': item['album']['name'],
            'album_id': item['album']['id'],
            'release_date': item['album']['release_date'],
            'popularity': item['popularity'],
            'duration_ms': item['duration_ms'],
            'explicit':item['explicit'],
            'track_id': item['id']
        }
        track_list.append(track_info)
    
    data = pd.DataFrame(track_list)
    data['duration_min'] = data['duration_ms']/60000
    data['release_date'] = pd.to_datetime(data['release_date'])

    return data

def extract_top_artist_data(spotify_client,time_range)->pd.DataFrame:
    '''
    Convert JSON file extracted from Spotify
    to dataframe for analysis
    '''
    top_artist_json = spotify_client.current_user_top_artists(limit = 50,time_range = time_range)
    artist_list = []
    for item in top_artist_json['items']:
        artist_info = {
            'artist_name':item['name'],
            'popularity':item['popularity'],
            'genres': ', '.join(item['genres']),
            'artist_id': item['id'],
            'followers': item['followers']['total'],
            # Gets the largest image if exists
            'image_url': item['images'][0]['url'] if item['images'] else None
        }
        artist_list.append(artist_info)
    data = pd.DataFrame(artist_list)
    return data

def get_albums_info(spotify_client,album_ids)->pd.DataFrame:
    '''
    Extract album information from top songs 
    '''
    album_data = [] 

    for i in range(0,len(album_ids),20):
        # Spotify API limit requests to 20 albums per call 
        batch = album_ids[i:i+20]
        albums = spotify_client.albums(batch)['albums']
        for album in albums:
            album_info = {
        'album_id': album['id'],
        'album_name': album['name'],
        'release_date': album['release_date'],
        'total_tracks': album['total_tracks'],
        'label': album.get('label', ''),
        'popularity': album.get('popularity'),
        'album_type': album['album_type'],
        'external_url': album['external_urls']['spotify'],
        'album_uri': album['uri'],
        'release_date_precision': album['release_date_precision'],
        'images': [image['url'] for image in album['images']] if album['images'] else [],  # List of images with varying sizes
        'genres': ', '.join(album['genres']) if 'genres' in album else '',  # Check if genres are present
        'artists': ', '.join([artist['name'] for artist in album['artists']]),  # Add artist names
        'track_ids': [track['id'] for track in album['tracks']['items']],  # Extract track IDs
        'available_markets': album.get('available_markets', []),  # Get available markets for the album
        }
            album_data.append(album_info)
    data = pd.DataFrame(album_data)
    return data

def get_tracks_info(spotify_client,track_ids)->pd.DataFrame:
    '''
    Extract additional information on top tracks
    '''
    track_data = [] 

    for i in range(0,len(track_ids),50):
        batch = track_ids[i:i+50]
        tracks = spotify_client.tracks(batch)['tracks']
        for track in tracks:
            track_info = {
                'track_id':track['id'],
                'track_name':track['name'],
                'popularity':track['popularity'],
                'explicit':track['explicit'],
                'artist_id':track['artists'][0]['id'],
                'album_id':track['album']['id'],
                # URL provide 30s preview of track
                'preview_url':track.get('preview_url',''),
                # URL of track page in spotify 
                'track_url':track['external_urls']['spotify'],
                # External Identifier to track songs globally
                'track_external_id':track['external_ids'].get('isrc',''),
                # Check whether track saved in user device
                'is_local':track['is_local'],
            }
            track_data.append(track_info)

    track_data = pd.DataFrame(track_data)
    return track_data

# Fetch Top Tracks

In [4]:
# Optional: remove existing cache for fresh login
if os.path.exists(".cache-my-music-app"):
    os.remove(".cache-my-music-app")

# Create the OAuth object
sp_oauth = SpotifyOAuth(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    redirect_uri=REDIRECT_URI,
    scope=["user-top-read", "user-library-read", "playlist-read-private"],
    cache_path=".cache-my-music-app"
)

# Create the Spotify client using the SpotifyOAuth instance directly
sp = spotipy.Spotify(auth_manager=sp_oauth)

# Test API call
try:
    results = sp.current_user_top_tracks(limit=10)
    for idx, item in enumerate(results['items']):
        print(f"{idx+1}. {item['name']} - {item['artists'][0]['name']}")
except spotipy.exceptions.SpotifyException as e:
    print(f"Error: {e}")

print("Fetching your top tracks...")
top_tracks = sp.current_user_top_tracks(
    limit=50, time_range='medium_term'
    )

short_term_track_data = extract_top_tracks_data(sp,'short_term')
medium_term_track_data = extract_top_tracks_data(sp,'medium_term')
long_term_track_data = extract_top_tracks_data(sp,'long_term')

1. CASANOVA POSSE - ALI
2. HER - MINNIE
3. Imaginary Friend - ITZY
4. Air - YEJI
5. I TRUST YOU - エミリア(CV:高橋李依)
6. Shopper - IU
7. Secret - IU
8. Whiplash - aespa
9. FREAK - YUQI
10. NEMONEMO - YENA
Fetching your top tracks...


In [5]:
long_term_track_data

,track_name,artist_name,album_name,album_id,release_date,popularity,duration_ms,explicit,track_id,duration_min
0,Nobody - from Kaiju No. 8,OneRepublic,Nobody (from Kaiju No. 8),3YmKf1haPAblZIrIPpuRTf,2024-04-12,70,153626,False,47N81NMkB488fuOwOC3Oip,2.560433
1,I GOT YOU,TWICE,I GOT YOU,6RZHj6L3NqrvcKeiBHQbjL,2024-02-02,59,173240,False,35dhwUoJNlxrPyEIJkfDnx,2.887333
2,FREAK,YUQI,YUQ1,7LYc8ngbhwha4aGJ5kVauc,2024-04-23,57,171080,False,6ERs9uORCo1MfV0m9ixCuv,2.851333
3,DIVE,TWICE,DIVE,0riep5s1F9ynpobjOSzbcr,2024-07-10,57,181860,False,5vK3WrTOp6rEoASx1jAsp1,3.031000
4,Fate,(G)I-DLE,2,0mC9MXPddkzggVsOXh5gd3,2024-01-29,62,161546,False,2vNPGH1x5ZwxTjlvzLCyc2,2.692433
5,Doughnut,TWICE,Celebrate,1nqz3cEjuvCMo8RHLBI9kM,2022-07-27,53,263680,False,65rmgd5uMb4Rgqb5dSiU0p,4.394667
6,MORE & MORE,TWICE,MORE & MORE,5KsduuDNWzt65TaHzmtciv,2020-06-01,63,199653,False,3omvXShuRPM3zbDpWYqf5g,3.327550
7,Red Rover,YUQI,YUQ1,7LYc8ngbhwha4aGJ5kVauc,2024-04-23,48,123320,False,4TQBHR8LcbBUv0LvLmn54H,2.055333
8,Full Moon Full Life,高橋あず美,Persona 3 Reload Original Soundtrack,20Bf2RVERC5Bc2eo3vyvJv,2024-04-24,65,293493,False,3Jl2LQmRwbXEF2lO1RTvxn,4.891550
9,YES or YES,TWICE,YES or YES,25VunQEW0x2W6ALND2Mh4g,2018-11-05,68,237680,False,26OVhEqFDQH0Ij77QtmGP9,3.961333


In [6]:
short_term_track_data

,track_name,artist_name,album_name,album_id,release_date,popularity,duration_ms,explicit,track_id,duration_min
0,Air,YEJI,Air,4ILxYaUCwfA9EJ36wPwTWz,2025-03-10,55,194800,False,2HhIndg75YiKjuUgGiMjSA,3.246667
1,Draw the Moon (feat. MIYAVI),MINNIE,"Webtoon <Myst, Might, Mayhem> OST Part. 2 Draw...",2dD84O2WUFNCjs963yWsbh,2025-03-21,45,210146,False,4B3JCEcAeTofpsfsEianeS,3.502433
2,Radio (Dum-Dum),YUQI,Radio (Dum-Dum),1jrJTnOMuLs5v0qTDTc0kR,2025-03-17,67,152373,False,0mXXjVVAhaasNXga2HMgJK,2.539550
3,I,TAEYEON,I - The 1st Mini Album,4e7kLQu7SKBUiMtV5WH3A1,2015-10-07,61,206038,False,5ZkITfPpcNPnyYGTibkO6m,3.433967
4,Invasion,YEJI,Air,4ILxYaUCwfA9EJ36wPwTWz,2025-03-10,40,167800,False,3ePablAj7jk2c1j5CKEtAv,2.796667
5,HAPPY,DAY6,Fourever,29pgfsXVV0cLsvfylWRZJ9,2024-03-18,64,189934,False,1k68vKHNQXU5CHqcM7Yp7N,3.165567
6,"Can’t Slow Me, No",YEJI,Air,4ILxYaUCwfA9EJ36wPwTWz,2025-03-10,38,178573,False,7EbaA7z7wP3G7xfYZVlVJS,2.976217
7,Shopper,IU,The Winning,08CvAj58nVMpq1Nw7T6maj,2024-02-20,55,215720,False,1c6kkrWnpy68eYDfBdxNtF,3.595333
8,SMILEY-Japanese Ver.- (feat.ちゃんみな),YENA,SMILEY-Japanese Ver.- (feat.ちゃんみな),405JFvqLG79ifF84IKlWJC,2023-08-07,39,176882,False,4RhkH4fGvvEzxvRlUcYH9L,2.948033
9,NEMONEMO,YENA,NEMONEMO,6FLiJ4318RtpA5lYWJt2cL,2024-09-30,67,178026,False,4UwsXGVppRRJpKBHy0mtyK,2.967100


# Fetch Top Artist

In [7]:
extract_top_artist_data(sp,'short_term')

,artist_name,popularity,genres,artist_id,followers,image_url
0,aespa,80,k-pop,6YVMFz59CuY7ngCxTxjpxE,8361617,https://i.scdn.co/image/ab6761610000e5ebf7a109...
1,YENA,56,k-pop,49muoiIu4uea4PO8vueUNN,807593,https://i.scdn.co/image/ab6761610000e5eb1135e1...
2,IU,70,"k-pop, k-ballad",3HqSLMAZ3g3d5poNaI7GOU,9135223,https://i.scdn.co/image/ab6761610000e5ebbd0642...
3,TAEYEON,67,"k-pop, k-ballad",3qNVuliS40BLgXGxhdBdqu,3064731,https://i.scdn.co/image/ab6761610000e5ebb5d9eb...
4,MINNIE,63,k-pop,2pHkxVNynHBwQHhGaoBIXX,761745,https://i.scdn.co/image/ab6761610000e5eb0134f9...
5,TWICE,79,k-pop,7n2Ycct7Beij7Dj7meI4X0,21665446,https://i.scdn.co/image/ab6761610000e5ebca6c14...
6,ITZY,69,k-pop,2KC9Qb60EaY0kW4eH68vr3,8351660,https://i.scdn.co/image/ab6761610000e5eb344806...
7,YEJI,61,k-pop,3skli1w2n0nOZ4qkDbvV2m,114620,https://i.scdn.co/image/ab6761610000e5eb485cfb...
8,YUQI,60,k-pop,22aCD8IrQZjcPgZw728QT6,833634,https://i.scdn.co/image/ab6761610000e5eb569026...
9,NMIXX,71,k-pop,28ot3wh4oNmoFOdVajibBl,3607044,https://i.scdn.co/image/ab6761610000e5ebb75519...


In [8]:
extract_top_artist_data(sp,'long_term')

,artist_name,popularity,genres,artist_id,followers,image_url
0,TWICE,79,k-pop,7n2Ycct7Beij7Dj7meI4X0,21665446,https://i.scdn.co/image/ab6761610000e5ebca6c14...
1,(G)I-DLE,73,k-pop,2AfmfGFbe0A0WsTYm0SDTx,10489801,https://i.scdn.co/image/ab6761610000e5eb7fd163...
2,Ed Sheeran,89,soft pop,6eUKZXaKkcviH0Ku9w2n3V,119923718,https://i.scdn.co/image/ab6761610000e5eb399444...
3,Taylor Swift,98,,06HL4z0CvFAxyc27GXpf02,135420613,https://i.scdn.co/image/ab6761610000e5ebe672b5...
4,IU,70,"k-pop, k-ballad",3HqSLMAZ3g3d5poNaI7GOU,9135223,https://i.scdn.co/image/ab6761610000e5ebbd0642...
5,ITZY,69,k-pop,2KC9Qb60EaY0kW4eH68vr3,8351660,https://i.scdn.co/image/ab6761610000e5eb344806...
6,aespa,80,k-pop,6YVMFz59CuY7ngCxTxjpxE,8361617,https://i.scdn.co/image/ab6761610000e5ebf7a109...
7,Against The Current,64,pop punk,6yhD1KjhLxIETFF7vIRf8B,528949,https://i.scdn.co/image/ab6761610000e5eb469aba...
8,MINNIE,63,k-pop,2pHkxVNynHBwQHhGaoBIXX,761745,https://i.scdn.co/image/ab6761610000e5eb0134f9...
9,IVE,75,k-pop,6RHTUrRF63xao58xh9FXYJ,5451531,https://i.scdn.co/image/ab6761610000e5eb538dad...


In [9]:
extract_top_artist_data(sp,'medium_term')

,artist_name,popularity,genres,artist_id,followers,image_url
0,aespa,80,k-pop,6YVMFz59CuY7ngCxTxjpxE,8361617,https://i.scdn.co/image/ab6761610000e5ebf7a109...
1,IU,70,"k-pop, k-ballad",3HqSLMAZ3g3d5poNaI7GOU,9135223,https://i.scdn.co/image/ab6761610000e5ebbd0642...
2,MINNIE,63,k-pop,2pHkxVNynHBwQHhGaoBIXX,761745,https://i.scdn.co/image/ab6761610000e5eb1a9a17...
3,YUQI,60,k-pop,22aCD8IrQZjcPgZw728QT6,833634,https://i.scdn.co/image/ab6761610000e5eb569026...
4,TWICE,79,k-pop,7n2Ycct7Beij7Dj7meI4X0,21665446,https://i.scdn.co/image/ab6761610000e5ebca6c14...
5,(G)I-DLE,73,k-pop,2AfmfGFbe0A0WsTYm0SDTx,10489801,https://i.scdn.co/image/ab6761610000e5eb7fd163...
6,ITZY,69,k-pop,2KC9Qb60EaY0kW4eH68vr3,8351660,https://i.scdn.co/image/ab6761610000e5eb344806...
7,IVE,75,k-pop,6RHTUrRF63xao58xh9FXYJ,5451531,https://i.scdn.co/image/ab6761610000e5eb538dad...
8,YENA,56,k-pop,49muoiIu4uea4PO8vueUNN,807593,https://i.scdn.co/image/ab6761610000e5eb1135e1...
9,Ed Sheeran,89,soft pop,6eUKZXaKkcviH0Ku9w2n3V,119923718,https://i.scdn.co/image/ab6761610000e5eb784daf...


# Extract Album Information

In [10]:
album_ids = short_term_track_data['album_id'].unique().tolist()
get_albums_info(sp,album_ids)

,album_id,album_name,release_date,total_tracks,label,popularity,album_type,external_url,album_uri,release_date_precision,images,genres,artists,track_ids,available_markets
0,4ILxYaUCwfA9EJ36wPwTWz,Air,2025-03-10,4,WM Japan,46,single,https://open.spotify.com/album/4ILxYaUCwfA9EJ3...,spotify:album:4ILxYaUCwfA9EJ36wPwTWz,day,[https://i.scdn.co/image/ab67616d0000b273dfd81...,,YEJI,"[6HSns0qPQQfgekCrBF0Dkf, 5xOUBAvwWYyuPmej4iRcV...",[JP]
1,2dD84O2WUFNCjs963yWsbh,"Webtoon <Myst, Might, Mayhem> OST Part. 2 Draw...",2025-03-21,2,GRAYGAMES,34,single,https://open.spotify.com/album/2dD84O2WUFNCjs9...,spotify:album:2dD84O2WUFNCjs963yWsbh,day,[https://i.scdn.co/image/ab67616d0000b273a17a4...,,MINNIE,"[4B3JCEcAeTofpsfsEianeS, 0tq66QKocw9dXtpN9E3T2O]","[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C..."
2,1jrJTnOMuLs5v0qTDTc0kR,Radio (Dum-Dum),2025-03-17,1,BMG Rights Management (US) LLC,55,single,https://open.spotify.com/album/1jrJTnOMuLs5v0q...,spotify:album:1jrJTnOMuLs5v0qTDTc0kR,day,[https://i.scdn.co/image/ab67616d0000b2737c841...,,YUQI,[0mXXjVVAhaasNXga2HMgJK],"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C..."
3,4e7kLQu7SKBUiMtV5WH3A1,I - The 1st Mini Album,2015-10-07,6,SM Entertainment,53,single,https://open.spotify.com/album/4e7kLQu7SKBUiMt...,spotify:album:4e7kLQu7SKBUiMtV5WH3A1,day,[https://i.scdn.co/image/ab67616d0000b273f531f...,,TAEYEON,"[5ZkITfPpcNPnyYGTibkO6m, 2DlBwbXR5mA8qhTuTMh6e...","[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C..."
4,29pgfsXVV0cLsvfylWRZJ9,Fourever,2024-03-18,7,Republic Records - DAY6,58,album,https://open.spotify.com/album/29pgfsXVV0cLsvf...,spotify:album:29pgfsXVV0cLsvfylWRZJ9,day,[https://i.scdn.co/image/ab67616d0000b27303099...,,DAY6,"[6je5cTal6PyeITNrOzkCoS, 1k68vKHNQXU5CHqcM7Yp7...","[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C..."
5,08CvAj58nVMpq1Nw7T6maj,The Winning,2024-02-20,5,EDAM Entertainment,55,single,https://open.spotify.com/album/08CvAj58nVMpq1N...,spotify:album:08CvAj58nVMpq1Nw7T6maj,day,[https://i.scdn.co/image/ab67616d0000b2735048e...,,IU,"[1c6kkrWnpy68eYDfBdxNtF, 0UTtK6hregIBOsefavRI2...","[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C..."
6,405JFvqLG79ifF84IKlWJC,SMILEY-Japanese Ver.- (feat.ちゃんみな),2023-08-07,2,DREAMUSIC,30,single,https://open.spotify.com/album/405JFvqLG79ifF8...,spotify:album:405JFvqLG79ifF84IKlWJC,day,[https://i.scdn.co/image/ab67616d0000b27393b11...,,YENA,"[4RhkH4fGvvEzxvRlUcYH9L, 5c0IvE05IS3ETib8Uv69mG]","[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C..."
7,6FLiJ4318RtpA5lYWJt2cL,NEMONEMO,2024-09-30,3,"Genie Music Corporation, Stone Music Entertain...",56,single,https://open.spotify.com/album/6FLiJ4318RtpA5l...,spotify:album:6FLiJ4318RtpA5lYWJt2cL,day,[https://i.scdn.co/image/ab67616d0000b27376bc3...,,YENA,"[4UwsXGVppRRJpKBHy0mtyK, 1T3HuFlijT5Uu4pZI60tg...","[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C..."
8,7qldKtsOWklzmwgll5NjCw,ˣ‿ˣ (SMiLEY),2022-01-17,5,"Genie Music Corporation, Stone Music Entertain...",50,single,https://open.spotify.com/album/7qldKtsOWklzmwg...,spotify:album:7qldKtsOWklzmwgll5NjCw,day,[https://i.scdn.co/image/ab67616d0000b273a435b...,,YENA,"[2l1igayfoeUbe8xjYTRGqi, 4zCIxSnVWpGNghERX4uWZ...","[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C..."
9,3vk3MR4IL2EMC4ksN5pCo4,Armageddon - The 1st Album,2024-05-27,10,SM Entertainment,59,album,https://open.spotify.com/album/3vk3MR4IL2EMC4k...,spotify:album:3vk3MR4IL2EMC4ksN5pCo4,day,[https://i.scdn.co/image/ab67616d0000b2731a956...,,aespa,"[2VdSktBqFfkW7y6q5Ik4Z4, 5eWcGfUCrVFMoYskyfkEP...",[JP]


# Extract additional track information

In [11]:
short_term_track_data.columns

Index(['track_name', 'artist_name', 'album_name', 'album_id', 'release_date',
       'popularity', 'duration_ms', 'explicit', 'track_id', 'duration_min'],
      dtype='object')

In [16]:
track_ids = short_term_track_data['track_id'].unique().tolist()
data = get_tracks_info(sp,track_ids)
data

,track_id,track_name,popularity,explicit,artist_id,album_id,preview_url,track_url,track_external_id,is_local
0,2HhIndg75YiKjuUgGiMjSA,Air,55,False,3skli1w2n0nOZ4qkDbvV2m,4ILxYaUCwfA9EJ36wPwTWz,None,https://open.spotify.com/track/2HhIndg75YiKjuU...,US5TA2500003,False
1,4B3JCEcAeTofpsfsEianeS,Draw the Moon (feat. MIYAVI),45,False,2pHkxVNynHBwQHhGaoBIXX,2dD84O2WUFNCjs963yWsbh,None,https://open.spotify.com/track/4B3JCEcAeTofpsf...,KRA382501968,False
2,0mXXjVVAhaasNXga2HMgJK,Radio (Dum-Dum),67,False,22aCD8IrQZjcPgZw728QT6,1jrJTnOMuLs5v0qTDTc0kR,None,https://open.spotify.com/track/0mXXjVVAhaasNXg...,KRA392500001,False
3,5ZkITfPpcNPnyYGTibkO6m,I,61,False,3qNVuliS40BLgXGxhdBdqu,4e7kLQu7SKBUiMtV5WH3A1,None,https://open.spotify.com/track/5ZkITfPpcNPnyYG...,KRA301500490,False
4,3ePablAj7jk2c1j5CKEtAv,Invasion,40,False,3skli1w2n0nOZ4qkDbvV2m,4ILxYaUCwfA9EJ36wPwTWz,None,https://open.spotify.com/track/3ePablAj7jk2c1j...,US5TA2500004,False
5,1k68vKHNQXU5CHqcM7Yp7N,HAPPY,64,False,5TnQc2N1iKlFjYD7CPGvFc,29pgfsXVV0cLsvfylWRZJ9,None,https://open.spotify.com/track/1k68vKHNQXU5CHq...,US5TA2400040,False
6,7EbaA7z7wP3G7xfYZVlVJS,"Can’t Slow Me, No",38,False,3skli1w2n0nOZ4qkDbvV2m,4ILxYaUCwfA9EJ36wPwTWz,None,https://open.spotify.com/track/7EbaA7z7wP3G7xf...,US5TA2500005,False
7,1c6kkrWnpy68eYDfBdxNtF,Shopper,55,False,3HqSLMAZ3g3d5poNaI7GOU,08CvAj58nVMpq1Nw7T6maj,None,https://open.spotify.com/track/1c6kkrWnpy68eYD...,KRA382401058,False
8,4RhkH4fGvvEzxvRlUcYH9L,SMILEY-Japanese Ver.- (feat.ちゃんみな),39,False,49muoiIu4uea4PO8vueUNN,405JFvqLG79ifF84IKlWJC,None,https://open.spotify.com/track/4RhkH4fGvvEzxvR...,JPN902301730,False
9,4UwsXGVppRRJpKBHy0mtyK,NEMONEMO,67,False,49muoiIu4uea4PO8vueUNN,6FLiJ4318RtpA5lYWJt2cL,None,https://open.spotify.com/track/4UwsXGVppRRJpKB...,KRMIM2465530,False


In [13]:
short_term_track_data

,track_name,artist_name,album_name,album_id,release_date,popularity,duration_ms,explicit,track_id,duration_min
0,Air,YEJI,Air,4ILxYaUCwfA9EJ36wPwTWz,2025-03-10,55,194800,False,2HhIndg75YiKjuUgGiMjSA,3.246667
1,Draw the Moon (feat. MIYAVI),MINNIE,"Webtoon <Myst, Might, Mayhem> OST Part. 2 Draw...",2dD84O2WUFNCjs963yWsbh,2025-03-21,45,210146,False,4B3JCEcAeTofpsfsEianeS,3.502433
2,Radio (Dum-Dum),YUQI,Radio (Dum-Dum),1jrJTnOMuLs5v0qTDTc0kR,2025-03-17,67,152373,False,0mXXjVVAhaasNXga2HMgJK,2.539550
3,I,TAEYEON,I - The 1st Mini Album,4e7kLQu7SKBUiMtV5WH3A1,2015-10-07,61,206038,False,5ZkITfPpcNPnyYGTibkO6m,3.433967
4,Invasion,YEJI,Air,4ILxYaUCwfA9EJ36wPwTWz,2025-03-10,40,167800,False,3ePablAj7jk2c1j5CKEtAv,2.796667
5,HAPPY,DAY6,Fourever,29pgfsXVV0cLsvfylWRZJ9,2024-03-18,64,189934,False,1k68vKHNQXU5CHqcM7Yp7N,3.165567
6,"Can’t Slow Me, No",YEJI,Air,4ILxYaUCwfA9EJ36wPwTWz,2025-03-10,38,178573,False,7EbaA7z7wP3G7xfYZVlVJS,2.976217
7,Shopper,IU,The Winning,08CvAj58nVMpq1Nw7T6maj,2024-02-20,55,215720,False,1c6kkrWnpy68eYDfBdxNtF,3.595333
8,SMILEY-Japanese Ver.- (feat.ちゃんみな),YENA,SMILEY-Japanese Ver.- (feat.ちゃんみな),405JFvqLG79ifF84IKlWJC,2023-08-07,39,176882,False,4RhkH4fGvvEzxvRlUcYH9L,2.948033
9,NEMONEMO,YENA,NEMONEMO,6FLiJ4318RtpA5lYWJt2cL,2024-09-30,67,178026,False,4UwsXGVppRRJpKBHy0mtyK,2.967100
